In [1]:
# ==========================================
# Task 2 Output for Task 4 Fusion (UPDATED)
# ==========================================

# -------------------------------
# Step 1: Environment Setup
# -------------------------------
import os
import random
import zipfile
import pickle
import glob

import numpy as np
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# -------------------------------
# Step 2: Load ONLY the fusion-aligned dataset
# -------------------------------
# We score the feature-engineered intake dataset, because that is what Task 3/4 will consume.
BASE_URL = "https://huggingface.co/datasets/ourafla/Mental-Health_Text-Classification_Dataset/resolve/main/"
feature_path = BASE_URL + "mental_health_feature_engineered.csv"

feature_df = pd.read_csv(feature_path)
print("feature_df loaded:", feature_df.shape)


feature_df loaded: (49612, 21)


In [4]:
# -------------------------------
# Step 3: Sanity checks for fusion join
# -------------------------------
# We want a stable key for joining Task 2 (weak) with Task 3 (strong) in Task 4.
# If your column name differs, adjust here.
KEY_COL_CANDIDATES = ["Unique_ID", "unique_id", "id", "ID"]

key_col = next((c for c in KEY_COL_CANDIDATES if c in feature_df.columns), None)
if key_col is None:
    raise ValueError(
        f"No join key found. Expected one of {KEY_COL_CANDIDATES}. "
        f"Columns available: {list(feature_df.columns)}"
    )

# Text column
TEXT_COL_CANDIDATES = ["text", "Text", "clean_text", "sentence"]
text_col = next((c for c in TEXT_COL_CANDIDATES if c in feature_df.columns), None)
if text_col is None:
    raise ValueError(
        f"No text column found. Expected one of {TEXT_COL_CANDIDATES}. "
        f"Columns available: {list(feature_df.columns)}"
    )

# Validate key integrity
# Drop rows where the key column is NaN to ensure clean fusion.
initial_rows = len(feature_df)
feature_df.dropna(subset=[key_col], inplace=True)
if len(feature_df) < initial_rows:
    print(f"Warning: Dropped {initial_rows - len(feature_df)} rows due to missing values in '{key_col}'.")

if feature_df[key_col].isna().any():
    raise ValueError(f"{key_col} contains missing values. Fix before generating fusion outputs.")

dupes = feature_df[key_col].duplicated().sum()
if dupes > 0:
    raise ValueError(f"{key_col} has {dupes} duplicates. Must be unique for clean 1:1 fusion joins.")

# Clean text
feature_df[text_col] = feature_df[text_col].fillna("").astype(str)
empty_text = (feature_df[text_col].str.strip() == "").sum()
if empty_text > 0:
    print(f"Warning: {empty_text} rows have empty text. Predictions may be weak for those.")

print(f"Using join key: {key_col} | Using text column: {text_col}")


Using join key: Unique_ID | Using text column: text


In [5]:
# -------------------------------
# Step 4: Unzip and load Task 2 baseline artifacts
# -------------------------------
zip_path = "task1_models.zip"          # your uploaded zip with baseline_lr_model.pkl & baseline_tfidf_vectorizer.pkl
extract_dir = "task2_models_unzipped"  # renamed for clarity

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Could not find {zip_path}. Upload it to Colab (Files panel) or place it in the working directory."
    )

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

print(f"Unzipped '{zip_path}' into '{extract_dir}'")

# Load exactly the correct files (NO 'pick the first pkl' behavior)
pkl_files = glob.glob(os.path.join(extract_dir, "**", "*.pkl"), recursive=True)

model_file = next((f for f in pkl_files if os.path.basename(f) == "baseline_lr_model.pkl"), None)
vectorizer_file = next((f for f in pkl_files if os.path.basename(f) == "baseline_tfidf_vectorizer.pkl"), None)

if model_file is None:
    raise FileNotFoundError("baseline_lr_model.pkl not found inside the zip. Check zip contents.")
if vectorizer_file is None:
    raise FileNotFoundError("baseline_tfidf_vectorizer.pkl not found inside the zip. Check zip contents.")

print("Loading model from:", model_file)
with open(model_file, "rb") as f:
    model = pickle.load(f)

print("Loading vectorizer from:", vectorizer_file)
with open(vectorizer_file, "rb") as f:
    vectorizer = pickle.load(f)


Unzipped 'task1_models.zip' into 'task2_models_unzipped'
Loading model from: task2_models_unzipped/task1_models/baseline_lr_model.pkl
Loading vectorizer from: task2_models_unzipped/task1_models/baseline_tfidf_vectorizer.pkl


In [15]:
# -------------------------------
# Step 5: Vectorize + predict (FULL weak modality vector, 4-class)
# -------------------------------

# Ensure join key is stable (avoid 0.0, 1.0 issues)
feature_df["Unique_ID"] = pd.to_numeric(feature_df["Unique_ID"], errors="raise").astype(int)

# Vectorize
X = vectorizer.transform(feature_df[text_col].tolist())

# Predict probabilities (required for fusion)
if not hasattr(model, "predict_proba"):
    raise AttributeError("Model must support predict_proba() to generate fusion probability vectors.")

probs = model.predict_proba(X)              # shape: (N, 4)
preds = probs.argmax(axis=1)
maxprob = probs.max(axis=1)

# Entropy (uncertainty) — useful gating signal for Task 4
eps = 1e-12
entropy = -(probs * np.log(probs + eps)).sum(axis=1)


In [16]:
# -------------------------------
# Step 6: Build fusion-ready output (named probs as proposal labels)
# -------------------------------
label_names = ["normal", "anxiety", "depression", "suicidal"]

# Safety check: proposal claims 4 classes
if probs.shape[1] != 4:
    raise ValueError(f"Expected 4 classes, got {probs.shape[1]}")

out = pd.DataFrame({
    "Unique_ID": feature_df["Unique_ID"].values,
    "task2_pred": preds.astype(int),
    "task2_maxprob": maxprob.astype(float),
    "task2_entropy": entropy.astype(float),
})

for j, name in enumerate(label_names):
    out[f"task2_p_{name}"] = probs[:, j].astype(float)

# Optional: verify probabilities sum ~ 1
pcols = [f"task2_p_{n}" for n in label_names]
row_sums = out[pcols].sum(axis=1).values
bad = np.sum(np.abs(row_sums - 1.0) > 1e-4)
if bad > 0:
    print(f"Warning: {bad} rows have probability sums not close to 1.")


In [17]:
# -------------------------------
# Step 7: Save + confirm
# -------------------------------
out_path = "task2_weak_modality_for_task4_named.csv"
out.to_csv(out_path, index=False)

print("\nSaved fusion-ready Task 2 outputs to:", out_path)
print("Output shape:", out.shape)
print("Columns:", out.columns.tolist())
print(out.head(3))


Saved fusion-ready Task 2 outputs to: task2_weak_modality_for_task4_named.csv
Output shape: (40012, 8)
Columns: ['Unique_ID', 'task2_pred', 'task2_maxprob', 'task2_entropy', 'task2_p_normal', 'task2_p_anxiety', 'task2_p_depression', 'task2_p_suicidal']
   Unique_ID  task2_pred  task2_maxprob  task2_entropy  task2_p_normal  \
0          0           0       0.969466       0.159236        0.969466   
1          1           1       0.979821       0.115603        0.013613   
2          2           1       0.993525       0.045679        0.002566   

   task2_p_anxiety  task2_p_depression  task2_p_suicidal  
0         0.022684            0.004817          0.003033  
1         0.979821            0.004435          0.002131  
2         0.993525            0.002890          0.001020  


In [18]:
print("num_classes_from_model =", probs.shape[1])
print("model.classes_ =", getattr(model, "classes_", None))
print("unique predicted labels =", np.unique(preds))

num_classes_from_model = 4
model.classes_ = [0 1 2 3]
unique predicted labels = [0 1 2 3]


In [19]:
out

,Unique_ID,task2_pred,task2_maxprob,task2_entropy,task2_p_normal,task2_p_anxiety,task2_p_depression,task2_p_suicidal
0,0,0,0.969466,0.159236,0.969466,0.022684,0.004817,0.003033
1,1,1,0.979821,0.115603,0.013613,0.979821,0.004435,0.002131
2,2,1,0.993525,0.045679,0.002566,0.993525,0.002890,0.001020
3,3,1,0.952814,0.225169,0.032486,0.952814,0.012750,0.001950
4,4,1,0.996295,0.027721,0.002369,0.996295,0.000880,0.000456
...,...,...,...,...,...,...,...,...
40007,53037,1,0.997932,0.015701,0.000153,0.997932,0.001843,0.000073
40008,53039,3,0.436774,1.131581,0.014089,0.278516,0.270621,0.436774
40009,53040,1,0.346587,1.310981,0.305920,0.346587,0.241066,0.106427
40010,53041,1,0.972079,0.153894,0.014790,0.972079,0.010050,0.003080


In [20]:
print("Duplicate IDs:", out.duplicated("Unique_ID").sum())
print("Prob sum min/max:", out[["task2_p_normal","task2_p_anxiety","task2_p_depression","task2_p_suicidal"]].sum(axis=1).min(),
      out[["task2_p_normal","task2_p_anxiety","task2_p_depression","task2_p_suicidal"]].sum(axis=1).max())

Duplicate IDs: 0
Prob sum min/max: 0.9999999999999996 1.0000000000000004


In [21]:
from google.colab import files

output_filename = "task2_weak_modality_for_task4_named.csv"
files.download(output_filename)
print(f"Downloading {output_filename}...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>